# خلاصه مراحل پیش‌پردازش داده‌های تایتانیک

## ۱. بارگذاری و بررسی اولیه داده
- خواندن داده ها با استفاده از کتابخانه `Pandas`
- نمایش اطلاعات کلی داده با `info()` و `describe()`
- شناسایی ستون‌های دارای مقادیر گمشده (Missing Values)

## ۲. مدیریت مقادیر گمشده

| ستون | روش پردازش |
|------|-------------|
| **Age** | پر کردن مقادیر گمشده با **میانگین (mean)**|
| **Cabin** | تبدیل به ویژگی باینری `Has_Cabin` (1 اگر مقدار داشت باشد، در غیر این صورت 0) و سپس حذف ستون اصلی |
| **Embarked** | پر کردن ۲ مقدار گمشده با **مد (mode)** یا شایع‌ترین مقدار |

## ۳. تبدیل متغیرهای دسته‌ای (Encoding)

- **Sex**: تبدیل به مقادیر عددی (`male` → 0, `female` → 1)
- **Embarked**: اعمال **One-Hot Encoding** (با حذف اولین سطح برای جلوگیری از تله dummy)
- **Name**: استخراج عنوان مسافر (`Title`) و نگاشت به اعداد (0 تا 4)

## ۴. مهندسی ویژگی (Feature Engineering)

ویژگی‌های جدید ساخته شده:

- **Title**: عنوان مسافر (Mr, Mrs, Miss, Master, Other)
- **FamilySize**: مجموع `SibSp + Parch + 1` (اندازه خانواده شامل خود شخص)
- **IsAlone**: نشان‌دهنده تنهایی مسافر (1 اگر `FamilySize == 1` باشد، در غیر این صورت 0)

## ۵. حذف ستون‌های غیرضروری

ستون‌های حذف شده از داده:

- `PassengerId` (از داده آموزش - در داده تست برای خروجی نگه داشته می‌شود)
- `Name` (پس از استخراج Title)
- `Ticket` (کد بلیط - تأثیر چندانی در پیش‌بینی ندارد)
- `Cabin` (پس از ساخت ویژگی `Has_Cabin`)

## ۶. مقیاس‌سازی (Scaling)

استفاده از `StandardScaler` برای نرمال‌سازی ویژگی‌های عددی:

- **Age**
- **Fare**
- **FamilySize**

```python
scaler = StandardScaler()
data[num_features] = scaler.fit_transform(X_train[num_features])
data[num_features] = scaler.transform(X_test[num_features])

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

## خواندن دیتای کشتی تایتانیک

In [2]:
data = pd.read_csv("titanic.csv")

In [3]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


##  آمار توصیفی

In [4]:
data.describe(include="all")

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,891.000000,891.000000,891.000000,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,NaN,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,NaN,"Braund, Mr. Owen Harris",male,NaN,NaN,NaN,347082,NaN,B96 B98,S
freq,NaN,NaN,NaN,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,2.308642,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,0.836071,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,1.000000,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,2.000000,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,3.000000,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,3.000000,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


## اطلاعات کلی دیتا
- انواع داده
- تعداد داده های Nan/Null

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


## بررسی دقیق تر داده های عددی گمشده
#### همانطور که میبینیم تعداد زیادی از داده های ستون کابین و مقداری از داده های ستون سن گمشده هستند

In [6]:
data.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

## مدیریت داده های گمشده

### حذف ستون 
- ستون کابین بخاطر حجم زیادی از داده های گمشده حذف میشود ولی بهتر است قبل از این پروسه یک ویژگی باینری از روی این ستون تولید شود

In [7]:
data["Has_Cabin"] = data["Cabin"].notna().astype(int)

In [8]:
data = data.drop(["Cabin"], axis=1)

### پر کردن داده های گمشده با میانگین و مد

In [9]:
si = SimpleImputer(missing_values=np.nan, strategy="mean")
age = np.reshape(data["Age"].values, (-1, 1))
data["Age"] = si.fit_transform(age)

In [10]:
embarked_mode = data["Embarked"].mode()[0]
data["Embarked"] = data["Embarked"].fillna(embarked_mode)

## تبدیل داده های Categorical به Numerical

#### لیبل گذاری برای جنسیت اشخاص

In [11]:
data["Sex"] = data["Sex"].map({"female": 0, "male": 1})

#### برای ویژگی Embarked از OHE استفاده میکنیم

In [12]:
data["Embarked"].unique()

array(['S', 'C', 'Q'], dtype=object)

In [13]:
le = LabelEncoder()
data["Embarked"] = le.fit_transform(data["Embarked"])
data

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,Has_Cabin
0,1,0,3,"Braund, Mr. Owen Harris",1,22.000000,1,0,A/5 21171,7.2500,2,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0,38.000000,1,0,PC 17599,71.2833,0,1
2,3,1,3,"Heikkinen, Miss. Laina",0,26.000000,0,0,STON/O2. 3101282,7.9250,2,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0,35.000000,1,0,113803,53.1000,2,1
4,5,0,3,"Allen, Mr. William Henry",1,35.000000,0,0,373450,8.0500,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",1,27.000000,0,0,211536,13.0000,2,0
887,888,1,1,"Graham, Miss. Margaret Edith",0,19.000000,0,0,112053,30.0000,2,1
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",0,29.699118,1,2,W./C. 6607,23.4500,2,0
889,890,1,1,"Behr, Mr. Karl Howell",1,26.000000,0,0,111369,30.0000,0,1


In [14]:
ct = ColumnTransformer([("Embarked", OneHotEncoder(), [10])], remainder="passthrough")
data = pd.DataFrame(ct.fit_transform(data))

In [15]:
data = pd.DataFrame(data.values, columns=["0", "1", "2", "PassengerId", "Survived", "Pclass", "Name", "Sex", "Age", "SibSp", "Parch", "Ticket", "Fare", "Has_Cabin"])
data

,0,1,2,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Has_Cabin
0,0.0,0.0,1.0,1,0,3,"Braund, Mr. Owen Harris",1,22.0,1,0,A/5 21171,7.25,0
1,1.0,0.0,0.0,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0,38.0,1,0,PC 17599,71.2833,1
2,0.0,0.0,1.0,3,1,3,"Heikkinen, Miss. Laina",0,26.0,0,0,STON/O2. 3101282,7.925,0
3,0.0,0.0,1.0,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0,35.0,1,0,113803,53.1,1
4,0.0,0.0,1.0,5,0,3,"Allen, Mr. William Henry",1,35.0,0,0,373450,8.05,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0.0,0.0,1.0,887,0,2,"Montvila, Rev. Juozas",1,27.0,0,0,211536,13.0,0
887,0.0,0.0,1.0,888,1,1,"Graham, Miss. Margaret Edith",0,19.0,0,0,112053,30.0,1
888,0.0,0.0,1.0,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",0,29.699118,1,2,W./C. 6607,23.45,0
889,1.0,0.0,0.0,890,1,1,"Behr, Mr. Karl Howell",1,26.0,0,0,111369,30.0,1


## استخراج و ایجاد ویژگی های کاربردی جدید از ویژگی های موجود

#### جداسازی ویژگی های جدید از اسم اشخاص

In [16]:
data['Title'] = data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

In [17]:
title_mapping = {
        'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3,
        'Dr': 4, 'Rev': 4, 'Col': 4, 'Major': 4, 'Mlle': 4,
        'Countess': 4, 'Ms': 4, 'Lady': 4, 'Jonkheer': 4,
        'Don': 4, 'Dona': 4, 'Mme': 4, 'Capt': 4, 'Sir': 4
    }
data['Title'] = data['Title'].map(title_mapping).fillna(4).astype(int)
data = data.drop("Name", axis=1)

#### ایجاد تعداد اغضای هر خانواده

In [18]:
data["FamilySize"] = data['SibSp'] + data['Parch'] + 1

In [19]:
data['IsAlone'] = (data['FamilySize'] == 1).astype(int)

## حذف ویژگی های غیرکاربردی

In [20]:
data.drop(['Ticket', 'PassengerId'], axis=1, inplace=True)

In [21]:
data

,0,1,2,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Has_Cabin,Title,FamilySize,IsAlone
0,0.0,0.0,1.0,0,3,1,22.0,1,0,7.25,0,0,2,0
1,1.0,0.0,0.0,1,1,0,38.0,1,0,71.2833,1,2,2,0
2,0.0,0.0,1.0,1,3,0,26.0,0,0,7.925,0,1,1,1
3,0.0,0.0,1.0,1,1,0,35.0,1,0,53.1,1,2,2,0
4,0.0,0.0,1.0,0,3,1,35.0,0,0,8.05,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0.0,0.0,1.0,0,2,1,27.0,0,0,13.0,0,4,1,1
887,0.0,0.0,1.0,1,1,0,19.0,0,0,30.0,1,1,1,1
888,0.0,0.0,1.0,0,3,0,29.699118,1,2,23.45,0,1,4,0
889,1.0,0.0,0.0,1,1,1,26.0,0,0,30.0,1,0,1,1


## Scaling Data

In [22]:
ss = StandardScaler()
data[['Age', 'Fare', 'FamilySize']] = ss.fit_transform(data[['Age', 'Fare', 'FamilySize']])

In [23]:
data

,0,1,2,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Has_Cabin,Title,FamilySize,IsAlone
0,0.0,0.0,1.0,0,3,1,-0.592481,1,0,-0.502445,0,0,0.059160,0
1,1.0,0.0,0.0,1,1,0,0.638789,1,0,0.786845,1,2,0.059160,0
2,0.0,0.0,1.0,1,3,0,-0.284663,0,0,-0.488854,0,1,-0.560975,1
3,0.0,0.0,1.0,1,1,0,0.407926,1,0,0.420730,1,2,0.059160,0
4,0.0,0.0,1.0,0,3,1,0.407926,0,0,-0.486337,0,0,-0.560975,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0.0,0.0,1.0,0,2,1,-0.207709,0,0,-0.386671,0,4,-0.560975,1
887,0.0,0.0,1.0,1,1,0,-0.823344,0,0,-0.044381,1,1,-0.560975,1
888,0.0,0.0,1.0,0,3,0,0.000000,1,2,-0.176263,0,1,1.299429,0
889,1.0,0.0,0.0,1,1,1,-0.284663,0,0,-0.044381,1,0,-0.560975,1


## جداسازی متغیرهای وابسته و مستقل

In [24]:
X = data.drop("Survived", axis=1)
y = data["Survived"]